<a href="https://colab.research.google.com/github/mneve920811/GitHubActionsDemo/blob/main/RetoEmpleados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

In [3]:
EmpleadosAttrition = pd.read_csv("empleadosRETO.csv")
EmpleadosAttrition.head()

,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,...,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,1,997,4,Male,...,22,4,3,80,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,1,178,2,Male,...,20,4,4,80,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,1,1780,2,Male,...,13,3,2,80,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,1,1118,2,Male,...,19,3,4,80,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,1,582,2,Male,...,12,3,4,80,15,2,4,6,7,Yes


In [4]:
EmpleadosAttrition.drop(columns=[
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours"
], inplace=True)

In [5]:
EmpleadosAttrition.head()

,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,4,Male,3,4,...,No,22,4,3,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,2,Male,3,2,...,No,20,4,4,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,2,Male,3,1,...,No,13,3,2,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,2,Male,3,3,...,No,19,3,4,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,2,Male,3,3,...,Yes,12,3,4,15,2,4,6,7,Yes


In [13]:
EmpleadosAttrition["HiringDate"] = pd.to_datetime(
    EmpleadosAttrition["HiringDate"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

EmpleadosAttrition["Year"] = EmpleadosAttrition["HiringDate"].dt.year.astype("Int64")

In [17]:
EmpleadosAttrition["HiringDate"] = pd.to_datetime(
    EmpleadosAttrition["HiringDate"],
    format="mixed",   # detecta automáticamente
    errors="coerce"
)

EmpleadosAttrition["Year"] = EmpleadosAttrition["HiringDate"].dt.year.astype("Int64")

In [18]:
EmpleadosAttrition["Year"].head()

,Year
0,2013
1,<NA>
2,<NA>
3,<NA>
4,2011


In [19]:
EmpleadosAttrition["YearsAtCompany"] = 2018 - EmpleadosAttrition["Year"]

In [20]:
EmpleadosAttrition.rename(columns={"DistanceFromHome": "DistanceFromHome_km"}, inplace=True)

In [21]:
EmpleadosAttrition["DistanceFromHome"] = EmpleadosAttrition["DistanceFromHome_km"].str.replace("km", "").astype(int)

In [22]:
EmpleadosAttrition.drop(columns=[
    "Year",
    "HiringDate",
    "DistanceFromHome_km"
], inplace=True)

In [23]:
SueldoPromedio = EmpleadosAttrition.groupby("Department")["MonthlyIncome"].mean()
SueldoPromedioDepto = SueldoPromedio.reset_index()
SueldoPromedioDepto

,Department,MonthlyIncome
0,Human Resources,6239.888889
1,Research & Development,6804.149813
2,Sales,7188.250000


In [24]:
scaler = MinMaxScaler()
EmpleadosAttrition["MonthlyIncome"] = scaler.fit_transform(
    EmpleadosAttrition[["MonthlyIncome"]]
)

In [25]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

columnas_categoricas = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "Attrition"
]

for col in columnas_categoricas:
    EmpleadosAttrition[col] = le.fit_transform(EmpleadosAttrition[col])

In [28]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in EmpleadosAttrition.columns:
    if EmpleadosAttrition[col].dtype == "object":
        EmpleadosAttrition[col] = le.fit_transform(EmpleadosAttrition[col].astype(str))

In [29]:
correlaciones = EmpleadosAttrition.corr()["Attrition"].sort_values(ascending=False)
correlaciones

,Attrition
Attrition,1.000000
OverTime,0.324777
MaritalStatus,0.187283
JobRole,0.078684
BusinessTravel,0.060677
Department,0.054236
DistanceFromHome,0.052732
EducationField,0.051184
PerformanceRating,-0.006471
NumCompaniesWorked,-0.009082


In [30]:
vars_seleccionadas = correlaciones[abs(correlaciones) >= 0.1].index

EmpleadosAttritionFinal = EmpleadosAttrition[vars_seleccionadas]
EmpleadosAttritionFinal.head()

,Attrition,OverTime,MaritalStatus,EnvironmentSatisfaction,YearsAtCompany,JobSatisfaction,JobInvolvement,MonthlyIncome,YearsInCurrentRole,Age,TotalWorkingYears,JobLevel
0,0,0,0,4,5,4,3,0.864269,4,50,32,4
1,0,0,0,2,<NA>,2,3,0.207340,2,36,7,2
2,1,0,2,2,<NA>,2,3,0.088062,0,21,1,1
3,0,0,2,2,<NA>,2,3,0.497574,6,52,18,3
4,1,1,1,2,7,3,3,0.664470,6,33,15,3


In [32]:
EmpleadosAttritionFinal = EmpleadosAttritionFinal.dropna()

In [33]:
pca = PCA()
EmpleadosAttritionPCA = pca.fit_transform(EmpleadosAttritionFinal)

In [34]:
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)
num_componentes = np.argmax(varianza_acumulada >= 0.80) + 1

num_componentes

np.int64(2)

In [35]:
for i in range(num_componentes):
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(**{
        f"C{i}": EmpleadosAttritionPCA[:, i]
    })

In [36]:
EmpleadosAttritionFinal.to_csv("EmpleadosAttritionFinal.csv", index=False)